In [ ]:
!pip install -q yt-dlp git+https://github.com/openai/whisper.git
!apt-get -qq install ffmpeg

In [ ]:
!pip install tiktoken

In [ ]:
import os
import subprocess
import whisper
import torch
import tiktoken
from transformers import AutoTokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import re


In [ ]:
MODEL_DIR = ""

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)

model.to(device)
model.eval()


In [ ]:
OUT_DIR = "/content/output"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:

def download_audio(youtube_url, out_dir="audio", cookies_file=None):
    os.makedirs(out_dir, exist_ok=True)

    cmd = [
        "yt-dlp",
        "-f", "bestaudio[ext=m4a]/bestaudio/best",
        "--extract-audio",
        "--audio-format", "mp3",
        "-o", os.path.join(out_dir, "%(id)s.%(ext)s"),
        youtube_url
    ]

    if cookies_file:
        cmd.insert(1, "--cookies")
        cmd.insert(2, cookies_file)

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        raise RuntimeError(result.stderr)

    video_id = youtube_url.split("v=")[-1].split("&")[0]

    for ext in ["mp3", "m4a", "webm"]:
        path = os.path.join(out_dir, f"{video_id}.{ext}")
        if os.path.exists(path):
            return path

    raise FileNotFoundError("Audio file not found")


def transcribe_audio(audio_path, language="uk"):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = whisper.load_model("medium", device=device)
    result = model.transcribe(audio_path, language=language)
    return result["segments"]


def smart_chunk_text(text, tokenizer, max_tokens=512):
    text = re.sub(r"\s+", " ", text)
    sentences = re.split(r'(?<=[.!?])\s+', text)

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        sentence_tokens = tokenizer.encode(sentence, add_special_tokens=False)

        if len(sentence_tokens) > max_tokens:
            for i in range(0, len(sentence_tokens), max_tokens):
                piece = tokenizer.decode(sentence_tokens[i:i + max_tokens])

                if current_chunk:
                    chunks.append(current_chunk.strip())
                    current_chunk = ""

                chunks.append(piece.strip())
            continue

        test_chunk = current_chunk + " " + sentence if current_chunk else sentence

        token_count = len(tokenizer.encode(test_chunk, add_special_tokens=False))

        if token_count > max_tokens:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence
        else:
            current_chunk = test_chunk

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


def segments_to_text(segments):
    return " ".join(seg["text"].strip() for seg in segments)


def generate_summary(text, tokenizer, model, device):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            num_beams=4,
            no_repeat_ngram_size=3,
            repetition_penalty=1.1,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def summarize_pipeline(text, tokenizer, model, device, chunk_tokens=512):
    chunks = smart_chunk_text(text, tokenizer, chunk_tokens)

    for i, chunk in enumerate(chunks, 1):
        print(f"\n=== TRANSCRIPT CHUNK {i} ===\n")
        print(chunk)

    level1_summaries = []

    for i, chunk in enumerate(chunks, 1):
        summary = generate_summary(chunk, tokenizer, model, device)
        level1_summaries.append(summary.strip())

        print(f"\n=== LEVEL 1 SUMMARY CHUNK {i} ===\n")
        print(summary)

    level1_text = " ".join(level1_summaries)

    level2_chunks = smart_chunk_text(level1_text, tokenizer, chunk_tokens)

    final_summaries = []

    for chunk in level2_chunks:
        summary = generate_summary(chunk, tokenizer, model, device)
        final_summaries.append(summary.strip())

    return " ".join(final_summaries)


def full_pipeline(youtube_url, tokenizer, model, device, cookies_file=None):
    audio_path = download_audio(youtube_url, cookies_file=cookies_file)
    segments = transcribe_audio(audio_path)
    text = segments_to_text(segments)

    summary = summarize_pipeline(
        text,
        tokenizer,
        model,
        device
    )

    return summary


youtube_url = ""

device = "cuda" if torch.cuda.is_available() else "cpu"


final_summary = full_pipeline(
    youtube_url,
    tokenizer,
    model,
    device
)

print("\n===== FINAL SUMMARY =====\n")
print(final_summary)